# Step 5 - Feature selection

Entrena un LightGBM rapido para ranking de importancia y selecciona top-N respetando `fixed_features`.


In [ ]:
# Imports y configuracion del proyecto
from project_config import init_notebook
config = init_notebook()

import utils
logger = utils.init_logger('info')


In [ ]:
import pandas as pd
from pathlib import Path
from analytics import df_split, get_features
from models.stacking.base_learners import LGBMBaseLearner

processed = Path(config['project_folder']) / config['data']['processed']['local_path']
train = pd.read_parquet(processed / 'train.parquet')
val = pd.read_parquet(processed / 'val.parquet')

target = config['model']['objective_column']
drop_cols = [c for c in config['data']['columns_to_drop'] if c in train.columns]
train = train.drop(columns=drop_cols, errors='ignore')
val = val.drop(columns=drop_cols, errors='ignore')

X_tr, y_tr = df_split(train, target)
X_va, y_va = df_split(val, target)


## 5.1 Entrenar LightGBM rapido para ranking


In [ ]:
learner = LGBMBaseLearner(params={'n_estimators': 500, 'learning_rate': 0.1})
learner.fit(X_tr, y_tr, X_va, y_va)

imp = pd.DataFrame({
    'feature_name': list(learner.get_feature_importance().keys()),
    'importance': list(learner.get_feature_importance().values()),
}).sort_values('importance', ascending=False).reset_index(drop=True)
imp.head(30)


## 5.2 Seleccionar top-N respetando fixed_features


In [ ]:
selected = get_features(imp, features_amount=40, fixed_features=config['model']['fixed_features'])
print(len(selected), 'features seleccionadas')
import json
(processed / 'selected_features.json').write_text(json.dumps(selected, indent=2))
